<img src=../images/gdd-logo.png width=300px align=right> 

# Custom Aggregating Functions

In this notebook, we have a look at customizing aggregation functions. 

This is useful whenever the default aggregations like `.max()`, `.sum()`, `.mean()`, etc. are not enough and you want a bit more flexibility in your data manipulation.

This notebook covers:

* [Exercise: Find the range](#1)
* [Using the custom function](#2)

First of all, let's load Pandas and the dataset again:

In [1]:
import pandas as pd

chickweight = (
    pd.read_csv('../data/chickweight.csv')
    .rename(str.lower, axis='columns')
)

Let's say you wanted to find the range of a column. Unfortunately there is no range method on a column:

In [ ]:
# chickweight['weight'].range()

Therefore you need to create a new function that does this. 
<a id='1'></a>
### <mark> Exercise: Find the range</mark>

1. In the cell below find the range of the `'weight'` column (without using groupby)

<details>
    
  <summary><span style="color:blue">Show hint</span></summary>
  
The range is given by **subtracting** the **min**imum from the **max**imum.
    
</details>

In [2]:
chickweight['weight'].max() - chickweight['weight'].min()

338

2. Now finish writing the function below called `my_range`. The function takes one column of data as an argument, so you should be able to call your function using `my_range(chickweight['weight'])`

<details>
    
  <summary><span style="color:blue">Show hint</span></summary>
  
The `column` will be `chickweight['weight']` when the function is called. So in the function you need to replace `chickweight['weight']` with `column`:
    
The series is what is given as an input, and it should return its range.


</details>

In [3]:
def my_range(dfc):
    return dfc.max() - dfc.min()

In [4]:
my_range(chickweight['weight'])

338

**Bonus:** Create the same function, but this time using a `lambda` expression.

In [6]:
rangev = lambda dfc: dfc.max() - dfc.min()

rangev(chickweight['weight'])

338

In [7]:
# %load ../answers/03_Aggregations/ex-range.py
# part 1

chickweight['weight'].max() - chickweight['weight'].min()

# part 2
def my_range(col):
    return col.max() - col.min()

my_range(chickweight['weight'])

# bonus
(lambda col: col.max() - col.min())(chickweight['weight'])

<a id='2'></a>
## Using the custom function

Now that you have created a function that can find the range, we can use this function in the `.agg()` method:

In [8]:
def get_range(col):
    return col.max() - col.min()

(
    chickweight
    .groupby('time')
    .agg(weight_range = ('weight', get_range))
)

,weight_range
time,
0,4
2,20
4,21
6,45
8,74
10,112
12,163
14,172
16,216


<mark>**Question:** Why don't you put `get_range` in quotations like you do with `'mean'`?</mark>

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
  Because `mean` is a built in method (eg. `df['col'].mean()`) that the `.agg()` method can look up within pandas. 
    
  `get_range` is a function that we need to reference directly.

</details>

You can also use a `lambda` function, to avoid having to define and name the function.

In [9]:
(
    chickweight
    .groupby('time')
    .agg(weight_range = ('weight', lambda col: col.max() - col.min()) )
)

,weight_range
time,
0,4
2,20
4,21
6,45
8,74
10,112
12,163
14,172
16,216


Putting it all together:

In [10]:
(
    chickweight
    .groupby(['time', 'diet'])
    .agg(num_chickens = ('rownum', 'count'),
         weight_mean = ('weight', 'mean'),
         weight_range = ('weight', lambda col: col.max() - col.min()),
    )
)

num_chickens  weight_mean  weight_range
time diet                                         
0    1               20    41.400000             4
     2               10    40.700000             4
     3               10    40.800000             3
     4               10    41.000000             3
2    1               20    47.250000            16
     2               10    49.400000             9
     3               10    50.400000             7
     4               10    51.800000             6
4    1               19    56.473684            15
     2               10    59.800000             7
     3               10    62.200000            10
     4               10    64.500000             8
6    1               19    66.789474            33
     2               10    75.400000            14
     3               10    77.900000            19
     4               10    83.900000            18
8    1               19    79.684211            55
     2               10    91.700000            59
     3               10    98.400000            43
     4               10   105.600000            33
10   1               19    93.052632            88
     2               10   108.500000            95
     3               10   117.100000            75
     4               10   126.000000            40
12   1               19   108.526316           114
     2               10   131.300000           147
     3               10   144.400000            98
     4               10   151.400000            57
14   1               18   123.388889           124
     2               10   141.900000           169
     3               10   164.500000           126
     4               10   161.800000            50
16   1               17   144.647059           156
     2               10   164.700000           203
     3               10   197.400000           152
     4               10   182.000000            77
18   1               17   158.941176           169
     2               10   187.700000           235
     3               10   233.100000           186
     4               10   202.900000           115
20   1               17   170.411765           197
     2               10   205.600000           242
     3               10   258.900000           205
     4                9   233.888889           106
21   1               16   177.750000           209
     2               10   214.700000           257
     3               10   270.300000           226
     4                9   238.555556           126